# v9: Cost-model capacity, per-vehicle spread, and averaging over k

**Version:** v9  
**Kaggle submission:** **#7** `sub07-2026-11-21-v9_k1` (public 19,600.79) and **#9** `sub09-2026-11-21-v9_em` (public 18,058.56) → `outputs/submission_v9_*.csv`; `v9_k4` not uploaded yet

**Hypothesis (A):** the cost model has never been tuned since v5 and may over-shrink; under-predicting expensive repairs is the costly direction (a unit of shortage costs 0.20, a unit of excess 0.02). Also test whether the spread of repair cost differs by vehicle. **(B):** with the shift k uncertain, choose the decision by averaging over a belief about k.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
HIST = {a: [f"s{i:03d}" for i in range(a, a + 10)] for a in (23, 37, 54, 80, 129, 154)}   # six hidden 10-bin histograms
HCOLS = {col for g in HIST.values() for col in g}

def shape(df):
    """Histogram-shape features: per-bin shares, mean/std bin, entropy, low/high-3 tail shares, log usage."""
    out = {"logU": np.log1p(df[HIST[54][:9]].sum(axis=1, min_count=9))}
    for a, g in HIST.items():
        g = g[:9] if a == 54 else g                        # test s063 is NaN after realignment
        x = df[g].to_numpy(float); tot = x.sum(1, keepdims=True)
        sh = np.where(tot > 0, x / np.where(tot > 0, tot, 1), np.nan); k = np.arange(len(g)); mean = (sh * k).sum(1)
        out.update({f"h{a}_sh{i}": sh[:, i] for i in range(len(g))})
        out[f"h{a}_mean"] = np.where(tot[:, 0] > 0, mean, np.nan)
        out[f"h{a}_std"] = np.where(tot[:, 0] > 0, np.sqrt(np.maximum((sh * k ** 2).sum(1) - mean ** 2, 0)), np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"h{a}_ent"] = np.where(tot[:, 0] > 0, -np.nansum(np.where(sh > 0, sh * np.log(sh), 0), 1), np.nan)
        out[f"h{a}_lo3"], out[f"h{a}_hi3"] = sh[:, :3].sum(1), sh[:, -3:].sum(1)
    return pd.DataFrame(out, index=df.index)

def ratios(df, cols):
    U = df[HIST[54][:9]].sum(axis=1, min_count=9).replace(0, np.nan)
    return df[[col for col in cols if col not in HCOLS]].div(U, axis=0).add_suffix("_perU")

In [4]:
def pipeline_costs(oofs, base="A", cm=CM, X=None, repeats=None):
    """Paired CV cost for several failure-probability candidates through the v5 pipeline
    (nested cost model fitted once per fold, Platt fitted on the other folds' OOF)."""
    X = dev[S] if X is None else X
    cost = {k: [] for k in oofs}
    for a, b in F:
        fa = a[y[a] == 1]
        m_b, resid = cost_model(cm, X.iloc[fa], np.log(c[fa]), X.iloc[b])
        for k, o in oofs.items():
            fl, bu = decide_pv(make_cal("platt").fit(o[a], y[a]).predict(o[b]), m_b, resid)
            cost[k].append(per_vehicle(c[b], fl, bu))
    return {k: np.array(v) for k, v in cost.items()}

def report(cost, base):
    A = cost[base]
    for k, x in cost.items():
        d = x - A; print(f"{k:12s} CV cost {x.mean():.4f} ± {x.std():.3f}   paired vs {base}: {d.mean():+.4f}   per fold {np.round(d, 3)}   better {int((d < 0).sum())}/{len(d)}")

### A1. Diagnostics and capacity of the cost model (repeated 3 × 5 CV on dev failures only; hold-out untouched)
Metrics: R², calibration slope of `log c ~ prediction` (1.0 = no shrinkage), bias in the lowest/highest predicted-cost quintile, and the **budget loss** (pinball loss at the 0.909 quantile).

In [5]:
f = dev[dev.failure == 1].reset_index(drop=True); Xf = pd.concat([f[S], shape(f)], axis=1)
lcf, cst = np.log(f.repair_cost.to_numpy()), f.repair_cost.to_numpy()
base = dict(subsample=0.8, colsample_bytree=0.5, min_child_weight=5, n_jobs=-1, random_state=SEED)
grid = {"v8: 200 trees d3": dict(n_estimators=200, learning_rate=0.03, max_depth=3), "500 trees d3": dict(n_estimators=500, learning_rate=0.03, max_depth=3),
        "1000 trees d3": dict(n_estimators=1000, learning_rate=0.03, max_depth=3), "500 trees d4": dict(n_estimators=500, learning_rate=0.03, max_depth=4),
        "1000 trees d2": dict(n_estimators=1000, learning_rate=0.03, max_depth=2), "2000 trees d2": dict(n_estimators=2000, learning_rate=0.03, max_depth=2)}
pins = {}
for name, g in grid.items():
    r2s, sl, lo, hi, pn = [], [], [], [], []
    for rep in range(3):
        m = np.zeros(len(f))
        for a, b in KFold(5, shuffle=True, random_state=rep).split(Xf): m[b] = xgb.XGBRegressor(**base, **g).fit(Xf.iloc[a], lcf[a]).predict(Xf.iloc[b])
        r = lcf - m; qq = pd.qcut(m, 5, labels=False); bq = np.exp(m + np.quantile(r, Q_STAR))
        r2s.append(r2_score(lcf, m)); sl.append(np.polyfit(m, lcf, 1)[0]); lo.append(r[qq == 0].mean()); hi.append(r[qq == 4].mean())
        pn.append(np.mean(UNDER * np.maximum(cst - bq, 0) + OVER * np.maximum(bq - cst, 0)))
    pins[name] = np.array(pn)
    print(f"{name:18s} R2 {np.mean(r2s):.3f}  slope {np.mean(sl):.2f}  bias low/high quintile {np.mean(lo):+.2f}/{np.mean(hi):+.2f}  budget loss {np.mean(pn):.2f}  (paired vs v8: {np.mean(pn - pins['v8: 200 trees d3']):+.2f})")

v8: 200 trees d3   R2 0.798  slope 1.10  bias low/high quintile -0.21/+0.15  budget loss 24.01  (paired vs v8: +0.00)


500 trees d3       R2 0.805  slope 1.07  bias low/high quintile -0.16/+0.12  budget loss 23.39  (paired vs v8: -0.62)


1000 trees d3      R2 0.805  slope 1.06  bias low/high quintile -0.14/+0.12  budget loss 23.21  (paired vs v8: -0.80)


500 trees d4       R2 0.801  slope 1.07  bias low/high quintile -0.15/+0.13  budget loss 23.38  (paired vs v8: -0.63)


1000 trees d2      R2 0.810  slope 1.05  bias low/high quintile -0.13/+0.08  budget loss 23.17  (paired vs v8: -0.84)


2000 trees d2      R2 0.807  slope 1.04  bias low/high quintile -0.11/+0.08  budget loss 23.24  (paired vs v8: -0.77)


### Is the spread vehicle-specific? (diagnostic on the v8 cost model's out-of-fold residuals)

In [6]:
from scipy.stats import spearmanr
m_oof = cross_val_predict(xgb.XGBRegressor(**CM), Xf, lcf, cv=inner); r_oof = lcf - m_oof
qq = pd.qcut(m_oof, 5, labels=False)
print("residual sd by predicted-cost quintile:", [round(float(r_oof[qq == i].std()), 3) for i in range(5)], "  overall", round(float(r_oof.std()), 3))
SC = dict(n_estimators=150, max_depth=2, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5, min_child_weight=10, random_state=SEED)
ah = cross_val_predict(xgb.XGBRegressor(**SC), Xf.assign(m=m_oof), np.abs(r_oof), cv=inner)
print(f"can the features predict |residual| out of sample? Spearman {spearmanr(np.abs(r_oof), ah)[0]:+.3f}")

residual sd by predicted-cost quintile: [0.73, 0.584, 0.737, 0.658, 0.861]   overall 0.74


can the features predict |residual| out of sample? Spearman +0.087


### A2. Full pipeline, total cost (3 × 5 repeated CV, paired vs the v8 cost model)
E = v8 cost model (200 trees, depth 3). A1 = 1000 trees, depth 2. A2 = A1 + a per-vehicle spread (a scale model on |out-of-fold residual|, residuals standardised).

In [7]:
X = pd.concat([dev[S], shape(dev)], axis=1)
CM9 = {**CM, "n_estimators": 1000, "max_depth": 2}
def cost_model_h(params, Xa, lc, Xb):
    m_in = cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner); r = lc - m_in
    mb = xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb)
    Za, Zb = Xa.assign(m=m_in), Xb.assign(m=mb)
    s_in = np.maximum(cross_val_predict(xgb.XGBRegressor(**SC), Za, np.abs(r), cv=inner), 0.05)
    z = r / s_in; z = z / z.std() * r.std()
    return mb, z, np.maximum(xgb.XGBRegressor(**SC).fit(Za, np.abs(r)).predict(Zb), 0.05) / np.mean(s_in)
cost = {v: [] for v in ("E", "A1", "A2")}; cov = {v: [] for v in cost}
for fs in (SEED, 1, 2):
    Fr = list(StratifiedKFold(5, shuffle=True, random_state=fs).split(np.zeros(len(y)), y)); oo = np.zeros(len(dev))
    for a, b in Fr: oo[b] = xgb.XGBClassifier(**PARAMS).fit(dev.loc[a, S], y[a]).predict_proba(dev.loc[b, S])[:, 1]
    for a, b in Fr:
        fa = a[y[a] == 1]; lc = np.log(c[fa]); q = make_cal("platt").fit(oo[a], y[a]).predict(oo[b]); fm = y[b] == 1
        for v, (prm, het) in {"E": (CM, False), "A1": (CM9, False), "A2": (CM9, True)}.items():
            if het: mb, r, sb = cost_model_h(prm, X.iloc[fa], lc, X.iloc[b])
            else: mb, r = cost_model(prm, X.iloc[fa], lc, X.iloc[b]); sb = None
            fl, bu = decide_pv(q, mb, r, scale=sb); cost[v].append(per_vehicle(c[b], fl, bu))
            s = np.ones(len(b)) if sb is None else sb
            cov[v].append(np.mean(np.log(c[b][fm]) <= mb[fm] + s[fm] * np.quantile(r, Q_STAR)))
E0 = np.array(cost["E"])
for v in cost:
    x = np.array(cost[v]); d = x - E0
    print(f"{v:3s} CV cost {x.mean():.4f} ± {x.std():.3f}   paired vs E {d.mean():+.4f} ± {d.std(ddof=1) / np.sqrt(15):.4f}   better {int((d < 0).sum())}/15   coverage {np.mean(cov[v]):.3f}")

E   CV cost 0.8664 ± 0.161   paired vs E +0.0000 ± 0.0000   better 0/15   coverage 0.919
A1  CV cost 0.8623 ± 0.150   paired vs E -0.0041 ± 0.0091   better 7/15   coverage 0.926
A2  CV cost 0.8780 ± 0.153   paired vs E +0.0116 ± 0.0097   better 4/15   coverage 0.902


### B. Averaging over the uncertain shift
For a fixed decision the expected cost is linear in $p$, so the Bayes decision under a belief about $k$ uses $\bar p=\sum_k w_k\,\text{shift}(p,k)$. The belief weights come from v7 (EM 5.7-6.5 likely biased up; public-score fit ~4; scale-free fit 2-4).

In [8]:
def em_odds_factor(p, pi_tr, it=300):
    """Saerens-Latinne-Decaestecker EM: odds factor k by which the test failure odds exceed the training odds."""
    pi = pi_tr
    for _ in range(it):
        a = pi / pi_tr * p
        pi = (a / (a + (1 - pi) / (1 - pi_tr) * (1 - p))).mean()
    return (pi / (1 - pi)) / (pi_tr / (1 - pi_tr))

In [9]:
oof = fit_oof(dev[S]); p_te = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1]
q_te = Platt().fit(oof, y).predict(p_te)
f_all = tr[tr.failure == 1]; Xt = pd.concat([te[S], shape(te)], axis=1); Xa = pd.concat([f_all[S], shape(f_all)], axis=1)
m_te, resid = cost_model(CM9, Xa, np.log(f_all.repair_cost.to_numpy()), Xt)        # the v9 cost model
M = np.exp(m_te[:, None] + np.quantile(resid, (np.arange(99) + .5) / 99)[None, :]); mu = M.mean(1)
def exp_cost(fl, b, p):
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return np.where(fl == 1, INSPECT + p * fail + (1 - p) * OVER * b, p * mu).mean()
belief = {2: .05, 3: .20, 4: .30, 5: .20, 6.5: .20, 8: .05}
pbar = sum(w * shift(q_te, k) for k, w in belief.items())
builds = {f"k={k}": decide_pv(shift(q_te, k), m_te, resid) for k in (3, 4, 5, 6.5)}; builds["avg belief"] = decide_pv(pbar, m_te, resid)
truths = (2, 3, 4, 5, 6.5, 8)
T = pd.DataFrame({f"true {kt}": {n: exp_cost(*fb, shift(q_te, kt)) for n, fb in builds.items()} for kt in truths}); R = T - T.min(axis=0)
print(f"belief-averaged p has mean {pbar.mean():.4f}; flag rate {builds['avg belief'][0].mean():.2%}")
print("regret / vehicle\n", R.round(4).to_string()); print("max regret over k in [2, 8]:", R.max(axis=1).round(3).to_dict())

belief-averaged p has mean 0.0385; flag rate 9.88%
regret / vehicle
             true 2  true 3  true 4  true 5  true 6.5  true 8
k=3         0.0000  0.0000  0.0211  0.0740    0.1944  0.3300
k=4         0.0647  0.0200  0.0000  0.0146    0.0819  0.1683
k=5         0.1409  0.0638  0.0137  0.0000    0.0274  0.0766
k=6.5       0.2805  0.1630  0.0747  0.0245    0.0000  0.0000
avg belief  0.1034  0.0409  0.0042  0.0029    0.0477  0.1130
max regret over k in [2, 8]: {'k=3': 0.33, 'k=4': 0.168, 'k=5': 0.141, 'k=6.5': 0.28, 'avg belief': 0.113}


### Files (v2 classifier, Platt, **v9 cost model**, shift k). The upload design isolates one change per comparison: `v9_k1` vs the known v5-Platt score = the new cost model alone; `v9_k4` vs `v9_k1` = the shift alone; `v9_em` vs `v9_k4` = whether k should be higher.

In [10]:
for k, name in ((1, "v9_k1"), (4, "v9_k4"), (6.53, "v9_em")):        # 6.53 = the EM estimate from v7
    fl, bu = decide_pv(shift(q_te, k), m_te, resid); print(f"k={k}: ", end=""); write_submission(te.id, fl, bu, name)

k=1: wrote outputs/submission_v9_k1.csv: flagged 6.40% of test vehicles
k=4: wrote outputs/submission_v9_k4.csv: flagged 9.56% of test vehicles
k=6.53: wrote outputs/submission_v9_em.csv: flagged 11.24% of test vehicles


## Results and decisions

**A. The cost model was over-shrunk:** it over-predicts cheap repairs (mean residual −0.21) and **under-predicts expensive ones (+0.15)**, the costly direction.

| Cost model | R² | Calibration slope | Budget loss |
|---|---|---|---|
| v8: 200 trees, depth 3 | 0.798 | 1.10 | 24.01 |
| 1000 trees, depth 3 | 0.805 | 1.06 | 23.21 |
| **1000 trees, depth 2** | **0.810** | **1.05** | **23.17** |
| 2000 trees, depth 2 | 0.807 | 1.04 | 23.24 |

- The budget loss improves in all 3 repeats (−0.84 for 1000 trees/depth 2). In **total cost** the gain is below what CV can resolve (A1 0.862 vs 0.866, −0.004 ± 0.009, 7/15 folds): its expected effect at the training failure rate is ~0.005/vehicle. On test, with 4-6× more failures, it should be worth ~150-250 public points. **A1 kept** (its own metrics improve consistently and the mechanism is clear).
- **A2 (per-vehicle spread) rejected:** residual sd varies only mildly across vehicles (0.58-0.86 by predicted-cost quintile, no trend) and the features predict |residual| only weakly (out-of-sample Spearman 0.09); modelling it made things worse (+0.012, 4/15) and pushed coverage below target (0.902).

**B. Averaging over k** behaves like an effective **k ≈ 4.4**: its belief-weighted cost is the lowest and its worst-case regret the smallest (0.113/vehicle over k in [2, 8], vs 0.141 for k=5 and 0.168 for k=4), but its average gain over k=4 or 5 is only ~0.003/vehicle. Its real use comes **after** the first probe uploads: the observed score differences update the belief about k (using the simulated likelihoods), and the final decision uses the posterior-averaged $p$.

**Caveats:** the capacity gain is invisible in total-cost CV (noise); all conclusions about k rest on our own model of the test depot, which has been off by 10-15% when predicting past public scores.

## Public results (21 Nov 2026)

| Upload | File | What it isolates | Public score |
|---|---|---|---|
| #6 (20 Nov) | `v5_platt` | v5 cost model, no shift (k=1) | 22,968.55 |
| #7 | `v9_k1` | v9 cost model, no shift (k=1) | **19,600.79** (−3,368 vs #6: the new cost model alone) |
| #8 | `v7_k4` | v5 cost model, shift k=4 (uploaded by mistake instead of `v9_k4`) | **19,067.17** (−3,902 vs #6: the shift alone) |
| #9 | `v9_em` | v9 cost model, shift k=6.53 (EM) | **18,058.56** (best so far) |

- **Both changes help, and they add up:** the cost model alone is worth ~15% and the shift alone ~17%; together (`v9_em`) the score is 21% below v5.
- The cost-model gain (−3,368) is larger than the ~−700 to −2,200 predicted from our own model, so on the new depot the cost model matters more than our estimate suggested.
- The shift effect is in line with k ≈ 4 (predicted −3,300 at k=4); the three scores together do not pin k down tightly, because differences of 1,000-2,000 are within the noise of a ~6,000-vehicle public subset. The next step is to update the belief about k from all scores jointly, then upload `v9_k4` and neighbours.